In [1]:
from pathlib import Path
import pandas as pd

# Execute o notebook a partir da pasta notebooks/
RAW = Path("../data/raw")

arquivos = []

for caminho in RAW.rglob("*"):
    if caminho.is_file():
        arquivos.append({
            "arquivo": caminho.name,
            "pasta": str(caminho.parent.relative_to(RAW)),
            "extensao": caminho.suffix.lower(),
            "tamanho_mb": round(caminho.stat().st_size / 1024**2, 2)
        })

inventario = pd.DataFrame(arquivos)

display(inventario)

,arquivo,pasta,extensao,tamanho_mb
0,sismigra_2023.zip,migracao,.zip,1.94
1,sismigra_2024.csv,migracao,.csv,55.28
2,sismigra_ativos.csv,migracao,.csv,365.85
3,sismigra_janeiro_agosto_2026.csv,migracao,.csv,36.61
4,sismigra_janeiro_dezembro_2025.csv,migracao,.csv,58.40
5,POP2026_20261029.ods,populacao,.ods,0.19
6,POP2026_20261029.xlsx,populacao,.xlsx,0.22
7,RAIS_CTPS_CAGED_2025_MOV.csv,trabalho,.csv,107.12
8,RAIS_CTPS_CAGED_2026_MOV.csv,trabalho,.csv,76.08
9,RAIS_CTPS_ESTOQUE_2025.csv,trabalho,.csv,49.38


In [2]:
# Os CSVs serão inspecionados em conjunto abaixo.
arquivos = sorted(RAW.rglob("*.csv"))
print(f"Arquivos CSV encontrados: {len(arquivos)}")

Arquivos CSV encontrados: 21


In [3]:
# Repetindo a inspeção em todos os arquivos CSV encontrados


resultado = []

for arquivo in arquivos:

    try:
        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="utf-8",
            nrows=5
        )
        codificacao = "utf-8"

    except UnicodeDecodeError:
        df = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin-1",
            nrows=5
        )
        codificacao = "latin-1"

    resultado.append({
        "arquivo": arquivo.name,
        "colunas": len(df.columns),
        "codificacao": codificacao,
        "nomes_colunas": ", ".join(df.columns)
    })

estrutura = pd.DataFrame(resultado)

display(estrutura)

,arquivo,colunas,codificacao,nomes_colunas
0,Tabela_Curso_Tecnico_2025_V2.csv,49,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
1,Tabela_Docente_2025_V2.csv,184,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
2,Tabela_Escola_2025_V2.csv,290,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
3,Tabela_Gestor_Escolar_2025_v2.csv,88,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
4,Tabela_Matricula_2025_V2.csv,263,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
5,Tabela_Turma_2025_V2.csv,218,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
6,MICRODADOS_CADASTRO_CURSOS_2024.CSV,223,latin-1,"NU_ANO_CENSO, NO_REGIAO, CO_REGIAO, NO_UF, SG_..."
7,MICRODADOS_ED_SUP_IES_2024.CSV,84,latin-1,"NU_ANO_CENSO, NO_REGIAO_IES, CO_REGIAO_IES, NO..."
8,sismigra_2024.csv,17,utf-8,"DATAREGISTRO, AMPARO, AMPARO_DESCRICAO, CLASSI..."
9,sismigra_ativos.csv,8,utf-8,"unidade_da_federacao, pais, amparo, descricao_..."


In [4]:
# Contando os registros de cada arquivo

# Lista para guardas os resultados 
quantidades = []

# Percorrer todos os arquivos CSV encontrados
for arquivo in arquivos: 

    # Lendo o arquivo CSV
    with open(arquivo, "rb") as f:
        # Contando o número de linhas
        total = sum(1 for linha in f) - 1  # Subtrai 1 para não contar o cabeçalho

    # Guardando o resultado
    quantidades.append({
        "arquivo": arquivo.name,
        "registros": total
    })
    
# Criando um DataFrame com os resultados
contagem = pd.DataFrame(quantidades)

display(contagem)

,arquivo,registros
0,Tabela_Curso_Tecnico_2025_V2.csv,32136
1,Tabela_Docente_2025_V2.csv,178772
2,Tabela_Escola_2025_V2.csv,214192
3,Tabela_Gestor_Escolar_2025_v2.csv,180540
4,Tabela_Matricula_2025_V2.csv,178766
5,Tabela_Turma_2025_V2.csv,178772
6,MICRODADOS_CADASTRO_CURSOS_2024.CSV,720349
7,MICRODADOS_ED_SUP_IES_2024.CSV,2561
8,sismigra_2024.csv,248167
9,sismigra_ativos.csv,3879289


In [5]:
# Juntar as duas tabelas pelo nome do arquivo 
inventario_final = pd.merge(contagem, estrutura, on="arquivo", how="left")

# Escolher as colunas que queremos mostrar no inventário final
inventario_final = inventario_final[["arquivo", "colunas", "registros", "codificacao"]]

display(inventario_final)

,arquivo,colunas,registros,codificacao
0,Tabela_Curso_Tecnico_2025_V2.csv,49,32136,latin-1
1,Tabela_Docente_2025_V2.csv,184,178772,latin-1
2,Tabela_Escola_2025_V2.csv,290,214192,latin-1
3,Tabela_Gestor_Escolar_2025_v2.csv,88,180540,latin-1
4,Tabela_Matricula_2025_V2.csv,263,178766,latin-1
5,Tabela_Turma_2025_V2.csv,218,178772,latin-1
6,MICRODADOS_CADASTRO_CURSOS_2024.CSV,223,720349,latin-1
7,MICRODADOS_ED_SUP_IES_2024.CSV,84,2561,latin-1
8,sismigra_2024.csv,17,248167,utf-8
9,sismigra_ativos.csv,8,3879289,utf-8


In [6]:
# Salvando a tabela final 

# Criar a pasta reports, caso ela não exista
pasta = Path("../reports")
pasta.mkdir(exist_ok=True)

# Salvando o inventário final em CSV
inventario_final.to_csv(
    pasta / "inventario_estrutural.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Inventário estrutural salvo em: {pasta / 'inventario_estrutural.csv'}")

Inventário estrutural salvo em: ..\reports\inventario_estrutural.csv
